# Setup & Imports

In [ ]:
!pip install -q Biopython sentencepiece transformers
!pip install -q https://github.com/debbiemarkslab/EVcouplings/archive/develop.zip

In [ ]:
import gc
import math
import os
import re
import random
import warnings
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.mixture import GaussianMixture
from torch.utils.data import Dataset, DataLoader
from transformers import T5EncoderModel, T5Tokenizer
from evcouplings.align import Alignment, read_fasta

pd.set_option('display.max_rows', None)
warnings.filterwarnings('ignore')

# Load ProtT5 Model

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False)
prot_model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
prot_model = prot_model.to(device).eval()
gc.collect()

# Utility Functions

In [ ]:
def normalize_minmax(data):
    """Min-max normalize a numpy array."""
    dmin, dmax = np.min(data), np.max(data)
    if dmax == dmin:
        return np.zeros_like(data)
    return (data - dmin) / (dmax - dmin)


def read_a3m(fileobj, inserts="first"):
    """
    Read an alignment in compressed a3m format and expand into a2m format.

    Parameters
    ----------
    fileobj : file-like object
        A3M alignment file.
    inserts : {'first', 'delete'}
        Keep inserts in first sequence, or delete any insert column.

    Returns
    -------
    OrderedDict
        Sequences in alignment (key: ID, value: sequence).
    """
    seqs = OrderedDict()

    for i, (seq_id, seq) in enumerate(read_fasta(fileobj)):
        seq = seq.replace(".", "")

        if inserts == "first":
            if i == 0:
                uppercase_cols = [
                    j for j, c in enumerate(seq)
                    if c == c.upper() or c == "-"
                ]
                gap_template = np.array(["."] * len(seq))
            else:
                uppercase_chars = [
                    c for c in seq if c == c.upper() or c == "-"
                ]
                filled = np.copy(gap_template)
                filled[uppercase_cols] = uppercase_chars
                seq = "".join(filled)

        elif inserts == "delete":
            seq = "".join(c for c in seq if c == c.upper() and c != ".")
        else:
            raise ValueError(f"Invalid option for inserts: {inserts}")

        seqs[seq_id] = seq

    return seqs

# MSA Processing

In [ ]:
def load_and_filter_msa(msa_filepath, min_identity=0.5, max_gap_frac=0.3,
                        fallback_identity=0.2, fallback_gap_frac=0.7,
                        min_sequences=15):
    """Load an MSA from an a3m file and filter by identity and gap fraction.

    If the initial filtering yields fewer than `min_sequences` sequences,
    falls back to more permissive thresholds.

    Parameters
    ----------
    msa_filepath : str
        Path to the .a3m MSA file.
    min_identity : float
        Minimum sequence identity to the first sequence.
    max_gap_frac : float
        Maximum fraction of gap characters per sequence.
    fallback_identity : float
        Relaxed identity threshold if too few sequences remain.
    fallback_gap_frac : float
        Relaxed gap threshold if too few sequences remain.
    min_sequences : int
        Minimum number of sequences before triggering fallback.

    Returns
    -------
    Alignment
        Filtered EVcouplings Alignment object.
    """
    with open(msa_filepath, "r") as infile:
        next(infile)  # skip header
        seqs_dict = read_a3m(infile, inserts="delete")

    aln = Alignment.from_dict(seqs_dict)
    print(f"Raw alignment: {aln.N} sequences, length {aln.L}")

    ident_perc = aln.identities_to(aln.matrix[0])
    gap_fracs = aln.count(axis="seq", char="-")

    def _filter(ident_thresh, gap_thresh):
        return [
            i for i in range(len(ident_perc))
            if ident_perc[i] > ident_thresh and gap_fracs[i] <= gap_thresh
        ]

    keep = _filter(min_identity, max_gap_frac)
    if len(keep) < min_sequences:
        print(f"Only {len(keep)} sequences after strict filter, using relaxed thresholds")
        keep = _filter(fallback_identity, fallback_gap_frac)

    aln_filtered = aln.select(sequences=keep)
    print(f"Filtered alignment: {aln_filtered.N} sequences")
    return aln_filtered

# ProtT5 Embedding

In [ ]:
def embed_sequences(sequences, tokenizer, model, device, batch_size=5):
    """Compute ProtT5 embeddings for a list of amino-acid sequences.

    Parameters
    ----------
    sequences : list[str]
        Space-separated AA sequences (non-standard AAs replaced with X).
    tokenizer : T5Tokenizer
    model : T5EncoderModel
    device : torch.device
    batch_size : int

    Returns
    -------
    list[np.ndarray]
        Per-residue embeddings, each of shape (seq_len, 1024).
    """
    embeddings = []

    for start in range(0, len(sequences), batch_size):
        batch = sequences[start:start + batch_size]
        ids = tokenizer.batch_encode_plus(batch, add_special_tokens=True, padding='longest')
        input_ids = torch.tensor(ids['input_ids']).to(device)
        attention_mask = torch.tensor(ids['attention_mask']).to(device)

        with torch.no_grad():
            output = model(input_ids=input_ids, attention_mask=attention_mask)
            emb_batch = output.last_hidden_state.cpu().numpy()

        for seq_num in range(len(emb_batch)):
            seq_len = (attention_mask[seq_num] == 1).sum().item()
            embeddings.append(emb_batch[seq_num][:seq_len - 1])  # exclude EOS token

        del attention_mask, input_ids
        gc.collect()

    return embeddings

# Feature Calculation (D2D)

In [ ]:
def _prepare_msa_sequences(aln):
    """Convert alignment matrix rows into space-separated, upper-cased sequences."""
    seqs = []
    for row_idx in range(len(aln)):
        chars = [c.upper() for c in aln.matrix[row_idx].tolist()]
        seq_str = " ".join(chars)
        seq_str = re.sub(r"[-.UZOB]", "X", seq_str)
        seqs.append(seq_str)
    return seqs


def _embed_single_mutation(mut_sequence, tokenizer, model, device, pool):
    """Embed a single mutant sequence and apply max-pooling."""
    chars = [c for c in mut_sequence]
    seq_str = " ".join(chars)
    seq_str = re.sub(r"[-.]", "X", seq_str)

    ids = tokenizer.batch_encode_plus([seq_str], add_special_tokens=True, padding='longest')
    input_ids = torch.tensor(ids['input_ids']).to(device)
    attention_mask = torch.tensor(ids['attention_mask']).to(device)

    with torch.no_grad():
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        emb = output.last_hidden_state.cpu().numpy()
        seq_len = (attention_mask == 1).sum().item()
        seq_emd = emb[:, :seq_len - 1, :]

    seq_emd = pool(torch.tensor(seq_emd)).numpy()

    del input_ids, attention_mask
    gc.collect()

    return seq_emd[0]  # shape: (L, reduced_dim)


def calculation_WT_MUT(uniprot, all_mutations, msa_path, tokenizer, model, device, pool):
    """Calculate D2D features: WT vs mutant log-likelihood differences.

    Parameters
    ----------
    uniprot : str
        UniProt accession.
    all_mutations : pd.DataFrame
        Must contain columns: 'mut_sequence', 'mutant'.
    msa_path : str
        Directory containing .a3m MSA files.
    tokenizer, model : ProtT5 tokenizer and encoder.
    device : torch.device
    pool : nn.MaxPool1d
        Pooling layer for dimensionality reduction.

    Returns
    -------
    tuple of (log_probs, mutation_labels, feature_differences)
    """
    print(f"Processing {uniprot}")

    # --- Load and filter MSA ---
    msa_file = os.path.join(msa_path, f"{uniprot}.a3m")
    if not os.path.isfile(msa_file):
        raise FileNotFoundError(f"MSA not found: {msa_file}")

    aln = load_and_filter_msa(msa_file)

    # --- Embed WT MSA sequences ---
    msa_sequences = _prepare_msa_sequences(aln)
    batch_size = 5 if aln.L < 501 else 1
    wt_embeddings = embed_sequences(msa_sequences, tokenizer, model, device, batch_size)

    arr_WT = np.array(wt_embeddings)
    arr_WT = pool(torch.tensor(arr_WT)).numpy()
    print(f"WT embedding shape: {arr_WT.shape}")

    # Save ProtT5 representations
    save_path = f'/content/drive/MyDrive/Colab Notebooks/clean_code/{uniprot}_MSA_protT5_20D.npz'
    np.savez_compressed(save_path, arr=arr_WT)

    # --- Fit per-position GMMs on WT ---
    n_positions = arr_WT.shape[1]
    gmm_models = {}
    thresholds = np.zeros(n_positions)
    wt_diffs = np.zeros(n_positions)

    for col in range(n_positions):
        col_data = arr_WT[:, col].reshape(-1, 1)
        gmm = GaussianMixture(n_components=1).fit(col_data)
        gmm_models[col] = gmm

        densities = gmm.score_samples(col_data)
        thresholds[col] = np.percentile(densities, 99)
        wt_diffs[col] = densities[0] - thresholds[col]

    # --- Compute features for each mutation ---
    log_probs_out = []
    mutations_out = []
    features_out = []

    for _, mut_row in all_mutations.iterrows():
        # Embed the mutant
        mut_emb = _embed_single_mutation(
            mut_row['mut_sequence'], tokenizer, model, device, pool
        )

        # Replace WT first row with mutant embedding
        arr_mut = arr_WT.copy()
        arr_mut[0] = mut_emb

        # Per-position mutant log-likelihood differences
        mut_diffs = np.zeros(n_positions)
        for col in range(n_positions):
            density = gmm_models[col].score_samples(arr_mut[0, col].reshape(1, -1))
            mut_diffs[col] = density.item() - thresholds[col]

        # Log-probability (quality of fit) at mutation position(s)
        mutant_label = str(mut_row['mutant'])
        if ':' in mutant_label:
            # Multiple mutations (e.g. "A10C:G20T")
            positions = [int(p[1:-1]) - 1 for p in mutant_label.split(':')]
            avg_log_prob = np.mean([
                np.mean(gmm_models[pos].score_samples(arr_WT[:, pos].reshape(-1, 1)))
                for pos in positions
            ])
        else:
            pos = int(mutant_label[1:-1]) - 1
            avg_log_prob = np.mean(
                gmm_models[pos].score_samples(arr_WT[:, pos].reshape(-1, 1))
            )

        mutations_out.append(f"{uniprot}_{mutant_label}")
        log_probs_out.append(round(avg_log_prob, 3))
        features_out.append((wt_diffs - mut_diffs).tolist())

    return log_probs_out, mutations_out, features_out

# D2Deep Classifier

In [ ]:
class Classifier2L(nn.Module):
    """Two hidden-layer binary classifier with BatchNorm and Dropout."""

    def __init__(self, hidden1, hidden2, dropout=0.0, num_features=2200):
        super().__init__()
        self.layer_1 = nn.Linear(num_features, hidden1)
        self.layer_2 = nn.Linear(hidden1, hidden2)
        self.layer_3 = nn.Linear(hidden2, 1)

        self.batchnorm1 = nn.BatchNorm1d(hidden1)
        self.batchnorm2 = nn.BatchNorm1d(hidden2)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.drop(self.relu(self.batchnorm1(self.layer_1(x))))
        x = self.drop(self.relu(self.batchnorm2(self.layer_2(x))))
        return self.layer_3(x)


class ClassifierDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = X_data
        self.y_data = y_data

    def __getitem__(self, index):
        return self.X_data[index], self.y_data[index]

    def __len__(self):
        return len(self.X_data)

# Prediction & Confidence

In [ ]:
def predict_protein(mutation_features, model, device, pad_length=2200, batch_size=32):
    """Run D2Deep inference on pre-computed features.

    Parameters
    ----------
    mutation_features : pd.DataFrame
        Must contain a 'features' column with lists of floats.
    model : Classifier2L
    device : torch.device
    pad_length : int
        Pad/truncate features to this length.
    batch_size : int
        Inference batch size (the original used 1 — very slow).

    Returns
    -------
    list[float]
        Sigmoid predictions, one per mutation.
    """
    # Pad features
    padded = []
    for feat in mutation_features['features']:
        padded.append(feat[:pad_length] + [0.0] * max(0, pad_length - len(feat)))

    X = torch.tensor(padded, dtype=torch.float32)
    print(f"Input shape: {X.shape}")

    # Dummy labels (required by DataLoader but unused)
    y = torch.zeros(len(X), dtype=torch.long)
    loader = DataLoader(
        ClassifierDataset(X, y),
        batch_size=batch_size,
        drop_last=False,  # BUG FIX: was drop_last=True, which silently drops the last batch
    )

    predictions = []
    model.eval()
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            probs = torch.sigmoid(logits).cpu().squeeze(-1).tolist()
            # squeeze handles both batch and single-element cases
            if isinstance(probs, float):
                probs = [probs]
            predictions.extend(probs)

    return [round(p, 4) for p in predictions]


def normalise_confidence(df):
    """Compute D2Deep confidence scores via min-max normalized log-probs.

    Modifies the DataFrame in-place and returns it.
    """
    log_min = df['log_prob'].min()
    log_max = df['log_prob'].max()

    if log_max == log_min:
        df['log_normalized'] = 0.0
    else:
        df['log_normalized'] = ((df['log_prob'] - log_min) / (log_max - log_min)).round(3)

    pred_high = df['D2Deep_prediction'] >= 0.5
    log_high = (df['log_normalized'] >= 0.5) & ~pred_high
    log_low = (df['log_normalized'] < 0.5) & ~pred_high

    df.loc[pred_high, 'D2Deep_confidence'] = df.loc[pred_high, 'log_normalized'].round(3)
    df.loc[log_high, 'D2Deep_confidence'] = (1 - df.loc[log_high, 'log_normalized']).abs().round(3)
    df.loc[log_low, 'D2Deep_confidence'] = (1 - df.loc[log_low, 'log_normalized'] * 1.3).round(3)

    return df

# Step 1: Compute D2D Features

In [ ]:
pool = nn.MaxPool1d(50)  # reduce ProtT5 from 1024 to ~20 dims per residue
msa_path = '/content/drive/MyDrive/my_colab/3rdYear/datasets/mmseq2/all_msas/'
output_path = '/content/drive/MyDrive/Colab Notebooks/clean_code/'
input_csv_path = ''  # path to saturation mutagenesis CSVs

In [ ]:
protein_list = ['P68431']

for pdb_id in protein_list:
    all_mutations = pd.read_csv(os.path.join(input_csv_path, f"{pdb_id}_all.csv"))

    if 'mutant' not in all_mutations.columns:
        all_mutations['mutant'] = (
            all_mutations['AA_orig']
            + all_mutations['position'].astype(str)
            + all_mutations['AA_targ']
        )

    name = all_mutations.iloc[0]['uniprot id']
    log_probs, mutations, features = calculation_WT_MUT(
        name, all_mutations, msa_path, tokenizer, prot_model, device, pool
    )
    gc.collect()

    df = pd.DataFrame({
        'mutation': mutations,
        'features': features,
        'log_prob': log_probs,
    })
    df.to_csv(os.path.join(output_path, f'D2D_features_{pdb_id}.csv'), index=False)
    print(f"{pdb_id}: {len(df)} mutations processed")

# Step 2: D2Deep Predictions

In [ ]:
path_D2Deep = '/content/drive/MyDrive/my_colab/3rdYear/GMM/model'

D2Deep_model = Classifier2L(hidden1=4096, hidden2=2048, dropout=0.3).to(device)
D2Deep_model.load_state_dict(torch.load(path_D2Deep, map_location=device))
D2Deep_model.eval()

In [ ]:
features_path = '/content/drive/MyDrive/Colab Notebooks/5thYear/D2D_features_revamp/IDP_fun/inputs/'
results_path = os.path.join(features_path, '..', 'results')
os.makedirs(results_path, exist_ok=True)

for filename in os.listdir(features_path):
    if not filename.endswith('.csv'):
        continue

    uniprot = filename.split('_')[2].split('.')[0]
    print(f"Processing {filename} ({uniprot})")

    df = pd.read_csv(os.path.join(features_path, filename))
    df['features'] = df['features'].apply(
        lambda x: [float(i) for i in x.strip('[]').split(',')]
    )

    # D2Deep predictions
    df['D2Deep_prediction'] = predict_protein(df, D2Deep_model, device)

    # Average feature value
    df['Average_features'] = df['features'].apply(lambda x: round(sum(x) / len(x), 3))

    # Confidence scores
    df = normalise_confidence(df)

    out_file = os.path.join(results_path, f'{uniprot}_d2d_results_confidence.csv')
    df.to_csv(out_file, index=False)
    print(f"  -> Saved {len(df)} results")